In [61]:
import re
import json

from IPython.display import display
import numpy as np
import pandas as pd
from itertools import product
from scipy.optimize import linear_sum_assignment
from sentence_transformers import SentenceTransformer

In [62]:
from itertools import product


def generate_schema_variations(
    base_schema,
    entity_name_map,
    attribute_name_map,
    include_original_entity=True,
    include_original_attribute=True,
):
    """
    Generate all schema variations for one script only (Latin OR Cyrillic).

    Parameters
    ----------
    base_schema : dict
        Example:
        {
            "SkiKarta": [("cena", "NUMERIC"), ("vazi_od", "DATE")]
        }

    entity_name_map : dict
        Example:
        {
            "SkiKarta": ["Karta", "Vleznica", ...]
        }

    attribute_name_map : dict
        Example:
        {
            "cena": ["price", "fee", ...]
        }

    include_original_entity : bool
        Whether to also include the original table name from base_schema
        as one of the possible variants.

    include_original_attribute : bool
        Whether to also include the original attribute name from base_schema
        as one of the possible variants.

    Returns
    -------
    list[dict]
        A list of schema dictionaries, each one being a full variation.
    """

    table_options = []  # list of tuples: (base_table_name, possible_table_names, attr_defs)

    for base_table_name, attrs in base_schema.items():
        # Entity/table name candidates
        entity_variants = list(entity_name_map.get(base_table_name, []))
        if include_original_entity and base_table_name not in entity_variants:
            entity_variants = [base_table_name] + entity_variants

        # Attribute candidates for this table
        attr_options = []
        for attr_name, sql_type in attrs:
            possible_attr_names = list(attribute_name_map.get(attr_name, []))
            if include_original_attribute and attr_name not in possible_attr_names:
                possible_attr_names = [attr_name] + possible_attr_names

            # store all possible concrete versions of this attribute
            attr_options.append([(candidate, sql_type) for candidate in possible_attr_names])

        table_options.append((entity_variants, attr_options))

    # Now build all full-schema combinations
    all_schema_variations = []

    # One choice of table name + one choice per attribute, for each table
    per_table_combinations = []
    for entity_variants, attr_options in table_options:
        # For this table:
        # choose 1 table name and 1 variant for each attribute
        table_variants = []
        for chosen_table_name in entity_variants:
            for chosen_attrs in product(*attr_options):
                table_variants.append((chosen_table_name, list(chosen_attrs)))
        per_table_combinations.append(table_variants)

    # Combine the chosen version of every table into one full schema
    for chosen_tables in product(*per_table_combinations):
        schema_variant = {}
        for table_name, attrs in chosen_tables:
            schema_variant[table_name] = attrs
        all_schema_variations.append(schema_variant)

    return all_schema_variations


def generate_all_script_separated_variations(
    base_schema,
    entity_names_lat,
    attribute_names_lat,
    entity_names_cyr,
    attribute_names_cyr,
    include_original_entity=True,
    include_original_attribute=True,
):
    """
    Generate Latin-only and Cyrillic-only schema variations separately.
    No Latin/Cyrillic mixing is done.

    Returns
    -------
    dict
        {
            "latin": [...],
            "cyrillic": [...]
        }
    """
    latin_variations = generate_schema_variations(
        base_schema=base_schema,
        entity_name_map=entity_names_lat,
        attribute_name_map=attribute_names_lat,
        include_original_entity=include_original_entity,
        include_original_attribute=include_original_attribute,
    )

    cyrillic_variations = generate_schema_variations(
        base_schema=base_schema,
        entity_name_map=entity_names_cyr,
        attribute_name_map=attribute_names_cyr,
        include_original_entity=include_original_entity,
        include_original_attribute=include_original_attribute,
    )

    return {
        "latin": latin_variations,
        "cyrillic": cyrillic_variations,
    }
import random


def generate_limited_schema_variations(
    base_schema,
    entity_name_map,
    attribute_name_map,
    max_schemas=10,
    include_original_entity=True,
    include_original_attribute=True,
    seed=None,
):
    """
    Generate up to `max_schemas` schema variations for one script only
    (Latin OR Cyrillic), without mixing scripts.

    Returns a list of schema dictionaries.
    """
    rng = random.Random(seed)
    seen = set()
    results = []
    attempts = 0
    max_attempts = max_schemas * 50  # avoid infinite loops if variation space is small

    while len(results) < max_schemas and attempts < max_attempts:
        attempts += 1
        schema_variant = {}

        for base_table_name, attrs in base_schema.items():
            # choose table/entity name
            entity_variants = list(entity_name_map.get(base_table_name, []))
            # if include_original_entity and base_table_name not in entity_variants:
            #     entity_variants = [base_table_name] + entity_variants

            chosen_table_name = rng.choice(entity_variants)

            # choose attribute names
            chosen_attrs = []
            for attr_name, sql_type in attrs:
                attr_variants = list(attribute_name_map.get(attr_name, []))
                if include_original_attribute and attr_name not in attr_variants:
                    attr_variants = [attr_name] + attr_variants

                chosen_attr_name = rng.choice(attr_variants)
                chosen_attrs.append((chosen_attr_name, sql_type))

            schema_variant[chosen_table_name] = chosen_attrs

        # make hashable signature to avoid duplicates
        signature = tuple(
            sorted(
                (table, tuple(attrs))
                for table, attrs in schema_variant.items()
            )
        )

        if signature not in seen:
            seen.add(signature)
            results.append(schema_variant)

    return results


def generate_latin_and_cyrillic_variations(
    base_schema,
    entity_names_lat,
    attribute_names_lat,
    entity_names_cyr,
    attribute_names_cyr,
    max_latin=10,
    max_cyrillic=10,
    include_original_entity=True,
    include_original_attribute=True,
    seed=None,
):
    """
    Generate a limited number of Latin-only and Cyrillic-only schemas.
    No mixing between scripts.
    """
    latin_variations = generate_limited_schema_variations(
        base_schema=base_schema,
        entity_name_map=entity_names_lat,
        attribute_name_map=attribute_names_lat,
        max_schemas=max_latin,
        include_original_entity=include_original_entity,
        include_original_attribute=include_original_attribute,
        seed=seed,
    )

    cyrillic_variations = generate_limited_schema_variations(
        base_schema=base_schema,
        entity_name_map=entity_names_cyr,
        attribute_name_map=attribute_names_cyr,
        max_schemas=max_cyrillic,
        include_original_entity=include_original_entity,
        include_original_attribute=include_original_attribute,
        seed=None if seed is None else seed + 1,
    )

    return {
        "latin": latin_variations,
        "cyrillic": cyrillic_variations,
    }
diagram2_schema = {
    "SkiSlope": ["name", "difficulty", "length", "lift_id"],
    "Skier": ["age", "name", "last_name", "skill_level"],
    "SkiLift": ["name", "capacity", "length", "status"],
    "Instructor": ["name", "surname", "license_level", "experience_years"],
    "SkiEquipment": ["type", "brand", "size", "status"],
    "SkiPass": ["valid_from", "valid_to", "price",  "skier_id"]
}
diagram1_schema = {
    "SkiKarta": ["cena", "vazi_od", "vazi_do", "skijac_id"],
    "SkiLift": ["ime", "kapacitet", "dolzina", "status"],
    "SkiStaza": ["ime", "tezina", "dolzina", "lift_id"],
    "Skijac": ["ime", "prezime", "vozrast", "nivo_na_veshtina"],
    "Oprema": ["tip", "golemina", "brend", "status"],
    "Instructor": ["ime", "prezime", "nivo_na_licenca", "godini_iskustvo"]
}
entity_names_lat={
    "SkiKarta": ["Karta","Vleznica","SkiVleznica","LiftPass","SkiLiftTicket","SkiTicket"],
    "SkiLift":["Lift","Zicara","Zhichara","Gondola"],
    "SkiStaza":["Stage","Staza","Teren","SkiStage","SkiTeren","Slope"],
    "Skijac":["Skier","Skijach","Natprevaruvach"],
    "Oprema":["Gear","SkiingGear","Equipment"],
    "Instructor":["Instruktor","Tutor","Instructor","Teacher"]
}

entity_names_cyr={
        "SkiKarta": ["Карта","Влезница","СкиВлезница","ЛифтДозвола","СкиЛифтКарта","СкиКарта"],
    "SkiLift":["Лифт","Жичара","Гондола"],
    "SkiStaza":["Стаза","Терен","СкиСтаза","Удолница"],
    "Skijac":["Скијач","Натпреварувач"],
    "Oprema":["Опрема","Скиопрема"],
    "Instructor":["Инструктор","Професор","Тренер"]
}

attribute_names_lat={
    "cena": ["price","fee","cost","payment","amount"],
    "vazi_od": ["from","date_from","start","start_date","begin_date","pochetok","pochetna_data","pocetok","pocnuva_na"],
    "vazi_do": ["untill","till","to","date_till","end","end_date","begin_date","kraj","zavrshuva_na","kraen_datum"],
    "skijac_id": ["ski_id","id","skier_id","id_skier"],
    "ime": ["name","naziv"],
    "kapacitet": ["capacity","max_weight","max_cap","max_load"],
    "dolzina": ["length","size","golemina"],
    "status":["sostojba","zafatenost"],
    "tezina": ["weight","level","hardness","rating"],
    "lift_id": ["lift_number","id"],
    "prezime": ["surname"],
    "vozrast": ["godini","age","starost"],
    "nivo_na_veshtina": ["ability","capabilities","level","sposobnost"],
    "tip":["type","version","verzija"],
    "golemina": ["size","dolzina","gabaritnost"],
    "brend": ["brand","marka"],
    "nivo_na_licenca": ["licence_level","dozvola_nivo","dozvola"],
    "godini_iskustvo": ["years_experience"]
}
attribute_names_cyr={
    "cena": ["цена"],
    "vazi_od": ["од","дата_од","почеток","почетна_дата","почнал_на"],
    "vazi_do": ["до","се_до","дата_до","крај","крајна_дата","завршува_на",],
    "skijac_id": ["ски_ид","скијач_ид","скијач_ид"],
    "ime": ["име","назив"],
    "kapacitet": ["капацитет","носивост","макс_капацитет"],
    "dolzina": ["должина","големина"],
    "status":["состојба","зафатеност"],
    "tezina": ["тежина","тезина","ниво","оценка"],
    "lift_id": ["лифт_ид","ид"],
    "prezime": ["презиме"],
    "vozrast": ["години","возраст","старост"],
    "nivo_na_veshtina": ["способност","можности","ниво"],
    "tip":["тип","верзија"],
    "golemina": ["големина","должина","габаритност"],
    "brend": ["бренд","марка"],
    "nivo_na_licenca": ["ниво_лиценца","дозвола_ниво","дозвола"],
    "godini_iskustvo": ["години_искуство"]
}
PROFESSOR_SCHEMA = {
    "SkiKarta": [
        ("cena",       "NUMERIC"),
        ("vazi_od",    "DATE"),
        ("vazi_do",    "DATE"),
        ("skijac_id",  "INTEGER"),
    ],
    "SkiLift": [
        ("ime",        "TEXT"),
        ("kapacitet",  "INTEGER"),
        ("dolzina",    "NUMERIC"),
        ("status",     "TEXT"),
    ],
    "SkiStaza": [
        ("ime",        "TEXT"),
        ("tezina",     "TEXT"),
        ("dolzina",    "NUMERIC"),
        ("lift_id",    "INTEGER"),
    ],
    "Skijac": [
        ("ime",                 "TEXT"),
        ("prezime",             "TEXT"),
        ("vozrast",             "INTEGER"),
        ("nivo_na_veshtina",    "TEXT"),
    ],
    "Oprema": [
        ("tip",        "TEXT"),
        ("golemina",   "TEXT"),
        ("brend",      "TEXT"),
        ("status",     "TEXT"),
    ],
    "Instructor": [
        ("ime",                "TEXT"),
        ("prezime",            "TEXT"),
        ("nivo_na_licenca",    "TEXT"),
        ("godini_iskustvo",    "INTEGER"),
    ],
}
schemas = generate_latin_and_cyrillic_variations(
    base_schema=PROFESSOR_SCHEMA,
    entity_names_lat=entity_names_lat,
    attribute_names_lat=attribute_names_lat,
    entity_names_cyr=entity_names_cyr,
    attribute_names_cyr=attribute_names_cyr,
    max_latin=20,
    max_cyrillic=20,
    seed=42
)

print("Latin schemas:", len(schemas["latin"]))
print("Cyrillic schemas:", len(schemas["cyrillic"]))

print("\nFirst Latin schema:")
print(schemas["latin"][0])

print("\nFirst Cyrillic schema:")
print(schemas["cyrillic"])



Latin schemas: 20
Cyrillic schemas: 20

First Latin schema:
{'SkiTicket': [('cena', 'NUMERIC'), ('vazi_od', 'DATE'), ('date_till', 'DATE'), ('ski_id', 'INTEGER')], 'Zicara': [('ime', 'TEXT'), ('kapacitet', 'INTEGER'), ('dolzina', 'NUMERIC'), ('zafatenost', 'TEXT')], 'SkiStage': [('ime', 'TEXT'), ('tezina', 'TEXT'), ('dolzina', 'NUMERIC'), ('lift_id', 'INTEGER')], 'Skier': [('naziv', 'TEXT'), ('prezime', 'TEXT'), ('godini', 'INTEGER'), ('sposobnost', 'TEXT')], 'SkiingGear': [('type', 'TEXT'), ('gabaritnost', 'TEXT'), ('marka', 'TEXT'), ('sostojba', 'TEXT')], 'Instruktor': [('ime', 'TEXT'), ('surname', 'TEXT'), ('dozvola_nivo', 'TEXT'), ('years_experience', 'INTEGER')]}

First Cyrillic schema:
[{'Карта': [('цена', 'NUMERIC'), ('почнал_на', 'DATE'), ('завршува_на', 'DATE'), ('ски_ид', 'INTEGER')], 'Жичара': [('име', 'TEXT'), ('kapacitet', 'INTEGER'), ('должина', 'NUMERIC'), ('зафатеност', 'TEXT')], 'Удолница': [('назив', 'TEXT'), ('tezina', 'TEXT'), ('големина', 'NUMERIC'), ('лифт_ид', 'I

In [63]:
# Format: { "TableName": [("attr_name", "SQL_TYPE"), ...] }

# ── Professor solution (diagram1) ──────────────────────────────────────────
PROFESSOR_SCHEMA = {
    "SkiKarta": [
        ("cena",       "NUMERIC"),
        ("vazi_od",    "DATE"),
        ("vazi_do",    "DATE"),
        ("skijac_id",  "INTEGER"),
    ],
    "SkiLift": [
        ("ime",        "TEXT"),
        ("kapacitet",  "INTEGER"),
        ("dolzina",    "NUMERIC"),
        ("status",     "TEXT"),
    ],
    "SkiStaza": [
        ("ime",        "TEXT"),
        ("tezina",     "TEXT"),
        ("dolzina",    "NUMERIC"),
        ("lift_id",    "INTEGER"),
    ],
    "Skijac": [
        ("ime",                 "TEXT"),
        ("prezime",             "TEXT"),
        ("vozrast",             "INTEGER"),
        ("nivo_na_veshtina",    "TEXT"),
    ],
    "Oprema": [
        ("tip",        "TEXT"),
        ("golemina",   "TEXT"),
        ("brend",      "TEXT"),
        ("status",     "TEXT"),
    ],
    "Instructor": [
        ("ime",                "TEXT"),
        ("prezime",            "TEXT"),
        ("nivo_na_licenca",    "TEXT"),
        ("godini_iskustvo",    "INTEGER"),
    ],
}

# ── Student solution (diagram2) ─────────────────────────────────────────────
STUDENT_SCHEMA = {
    "SkiSlope": [
        ("name",        "TEXT"),
        ("difficulty",  "TEXT"),
        ("length",      "NUMERIC"),
        ("lift_id",     "INTEGER"),
    ],
    "Skier": [
        ("age",          "INTEGER"),
        ("name",         "TEXT"),
        ("last_name",    "TEXT"),
        ("skill_level",  "TEXT"),
    ],
    "SkiLift": [
        ("name",       "TEXT"),
        ("capacity",   "INTEGER"),
        ("length",     "NUMERIC"),
        ("status",     "TEXT"),
    ],
    "Instructor": [
        ("name",              "TEXT"),
        ("surname",           "TEXT"),
        ("license_level",     "TEXT"),
        ("experience_years",  "INTEGER"),
    ],
    "SkiEquipment": [
        ("type",    "TEXT"),
        ("brand",   "TEXT"),
        ("size",    "TEXT"),
        ("status",  "TEXT"),
    ],
    "SkiPass": [
        ("valid_from",  "DATE"),
        ("valid_to",    "DATE"),
        ("price",       "NUMERIC"),
        ("skier_id",    "INTEGER"),
    ],
}

print(f'Professor tables : {list(PROFESSOR_SCHEMA.keys())}')
print(f'Student tables   : {list(STUDENT_SCHEMA.keys())}')

Professor tables : ['SkiKarta', 'SkiLift', 'SkiStaza', 'Skijac', 'Oprema', 'Instructor']
Student tables   : ['SkiSlope', 'Skier', 'SkiLift', 'Instructor', 'SkiEquipment', 'SkiPass']


In [64]:
# Minimum cosine similarity to consider a pair matched (0.0 – 1.0)
TABLE_MATCH_THRESHOLD = 0.40
ATTR_MATCH_THRESHOLD  = 0.35   # slightly lower now — type signal compensates

# Signal weights for TABLE matching (must sum to 1.0)
W_NAME   = 0.40   # table name embedding similarity
W_ATTRS  = 0.40   # attribute-set embedding similarity
W_STRUCT = 0.10   # structural similarity (attribute count ratio)
W_TYPES  = 0.10   # type-distribution similarity

# Weight of type compatibility in ATTRIBUTE matching (0.0 = ignore types)
W_ATTR_NAME = 0.65   # name embedding similarity
W_ATTR_TYPE = 0.35   # SQL type compatibility bonus

# Model name
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"  # ~120 MB, fast

print('Config loaded ✓')



Config loaded ✓


In [65]:
# SQL type normalisation map — collapse vendor-specific types into canonical groups
TYPE_GROUPS = {
    'integer': {'int', 'integer', 'bigint', 'smallint', 'tinyint', 'serial', 'number'},
    'text':    {'varchar', 'nvarchar', 'char', 'nchar', 'text', 'string', 'clob'},
    'decimal': {'decimal', 'numeric', 'float', 'double', 'real', 'money'},
    'date':    {'date'},
    'datetime':{'datetime', 'timestamp', 'time'},
    'boolean': {'boolean', 'bool', 'bit'},
    'binary':  {'blob', 'binary', 'varbinary', 'bytea'},
}

# Build reverse map: raw_type -> canonical_group
_TYPE_LOOKUP = {raw: group for group, raws in TYPE_GROUPS.items() for raw in raws}


def normalise_type(t: str) -> str:
    """Normalise a SQL type string to a canonical group name."""
    base = t.lower().split('(')[0].strip()   # strip length: VARCHAR(255) -> varchar
    return _TYPE_LOOKUP.get(base, base)       # unknown types kept as-is


def type_compatible(t1: str, t2: str) -> float:
    """
    Returns a compatibility score between two SQL types:
      1.0  — same canonical group  (INTEGER vs BIGINT)
      0.5  — related groups        (integer vs decimal — both numeric)
      0.0  — incompatible          (text vs date)
    """
    NUMERIC = {'integer', 'decimal'}
    TEMPORAL = {'date', 'datetime'}
    n1, n2 = normalise_type(t1), normalise_type(t2)
    if n1 == n2:
        return 1.0
    if n1 in NUMERIC and n2 in NUMERIC:
        return 0.5
    if n1 in TEMPORAL and n2 in TEMPORAL:
        return 0.5
    return 0.0


def type_distribution_similarity(prof_attrs: list, stud_attrs: list) -> float:
    """
    Compare the bag-of-types of two tables.
    e.g. [INTEGER x3, VARCHAR x2] vs [INTEGER x3, VARCHAR x2] -> 1.0
    Used as a table-level signal (not per-attribute).
    """
    from collections import Counter
    def type_counts(attrs):
        return Counter(normalise_type(t) for _, t in attrs)
    pc, sc = type_counts(prof_attrs), type_counts(stud_attrs)
    all_types = set(pc) | set(sc)
    if not all_types:
        return 1.0
    overlap = sum(min(pc.get(tp, 0), sc.get(tp, 0)) for tp in all_types)
    total   = sum(max(pc.get(tp, 0), sc.get(tp, 0)) for tp in all_types)
    return overlap / total if total > 0 else 0.0


# ── Unchanged helpers ──────────────────────────────────────────────────────

def tokenize(name: str) -> list:
    name = re.sub(r'([a-z])([A-Z])', r'\1 \2', name)
    tokens = re.split(r'[_\-\s]+', name)
    return [t.strip() for t in tokens if t.strip()]


def readable(name: str) -> str:
    return ' '.join(tokenize(name)).lower()


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0


def embed(model, texts: list) -> np.ndarray:
    return model.encode(texts, convert_to_numpy=True, show_progress_bar=False)


def attribute_set_similarity(model, prof_attrs: list, stud_attrs: list) -> float:
    """Average best-match name similarity across attribute sets (ignores types)."""
    if not prof_attrs or not stud_attrs:
        return 0.0
    pe = embed(model, [readable(a) for a, _ in prof_attrs])
    se = embed(model, [readable(a) for a, _ in stud_attrs])
    sim = np.array([[cosine(p, s) for s in se] for p in pe])
    return float(sim.max(axis=1).mean())


def structural_similarity(prof_attrs: list, stud_attrs: list) -> float:
    a, b = len(prof_attrs), len(stud_attrs)
    if a == 0 and b == 0:
        return 1.0
    return min(a, b) / max(a, b) if max(a, b) > 0 else 0.0


print('Helper functions defined ✓')

Helper functions defined ✓


In [66]:
print(f'Loading "{MODEL_NAME}" ...')
model = SentenceTransformer(MODEL_NAME)
print('Model ready ✓')

Loading "paraphrase-multilingual-MiniLM-L12-v2" ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13919.00it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model ready ✓


In [67]:
def build_table_similarity_matrix(model, professor: dict, student: dict):
    prof_tables = list(professor.keys())
    stud_tables = list(student.keys())

    prof_name_embs = embed(model, [readable(t) for t in prof_tables])
    stud_name_embs = embed(model, [readable(t) for t in stud_tables])

    n_p, n_s = len(prof_tables), len(stud_tables)
    matrix = np.zeros((n_p, n_s))

    for i, j in product(range(n_p), range(n_s)):
        pt, st = prof_tables[i], stud_tables[j]
        name_sim   = cosine(prof_name_embs[i], stud_name_embs[j])
        attr_sim   = attribute_set_similarity(model, professor[pt], student[st])
        struct_sim = structural_similarity(professor[pt], student[st])
        type_sim   = type_distribution_similarity(professor[pt], student[st])  # NEW
        matrix[i, j] = (W_NAME   * name_sim
                       + W_ATTRS  * attr_sim
                       + W_STRUCT * struct_sim
                       + W_TYPES  * type_sim)                                  # NEW

    return matrix, prof_tables, stud_tables

for shema in schemas["latin"]: #latin vo cyrillic
    sim_matrix, prof_tables, stud_tables = build_table_similarity_matrix(
        model, PROFESSOR_SCHEMA, shema
    )


    df_sim = pd.DataFrame(sim_matrix, index=prof_tables, columns=stud_tables)
    print('Table similarity matrix (higher = more similar):\n')

    display(df_sim.style
        .background_gradient(cmap='YlGn', vmin=0, vmax=1)
        .format('{:.3f}')
    )


Table similarity matrix (higher = more similar):



,SkiTicket,Zicara,SkiStage,Skier,SkiingGear,Instruktor
SkiKarta,0.943,0.559,0.751,0.779,0.747,0.488
SkiLift,0.706,0.648,0.810,0.756,0.717,0.531
SkiStaza,0.762,0.640,0.959,0.809,0.764,0.531
Skijac,0.734,0.622,0.787,0.933,0.835,0.607
Oprema,0.460,0.594,0.526,0.577,0.637,0.639
Instructor,0.399,0.517,0.489,0.590,0.487,0.920


Table similarity matrix (higher = more similar):



,Vleznica,Gondola,Teren,Skier,Gear,Instructor
SkiKarta,0.656,0.536,0.579,0.779,0.551,0.474
SkiLift,0.489,0.586,0.631,0.756,0.621,0.505
SkiStaza,0.496,0.600,0.713,0.810,0.579,0.517
Skijac,0.513,0.581,0.637,0.901,0.619,0.615
Oprema,0.584,0.546,0.686,0.577,0.876,0.623
Instructor,0.459,0.479,0.563,0.536,0.551,0.990


Table similarity matrix (higher = more similar):



...

,Karta,Gondola,Slope,Skier,Gear,Tutor
SkiKarta,0.757,0.457,0.631,0.787,0.572,0.473
SkiLift,0.493,0.523,0.686,0.760,0.576,0.516
SkiStaza,0.539,0.522,0.800,0.834,0.570,0.533
Skijac,0.536,0.515,0.682,0.942,0.646,0.590
Oprema,0.589,0.488,0.540,0.584,0.784,0.619
Instructor,0.480,0.436,0.469,0.576,0.564,0.895


Table similarity matrix (higher = more similar):



,SkiTicket,Gondola,SkiStage,Skier,SkiingGear,Tutor
SkiKarta,0.872,0.542,0.746,0.777,0.726,0.378
SkiLift,0.709,0.587,0.801,0.762,0.718,0.480
SkiStaza,0.775,0.597,0.911,0.810,0.757,0.465
Skijac,0.736,0.581,0.793,0.893,0.819,0.533
Oprema,0.466,0.564,0.540,0.587,0.652,0.565
Instructor,0.404,0.481,0.481,0.539,0.475,0.843


Table similarity matrix (higher = more similar):



,Vleznica,Zicara,SkiTeren,Skijach,Gear,Tutor
SkiKarta,0.558,0.555,0.770,0.695,0.551,0.414
SkiLift,0.427,0.579,0.794,0.695,0.586,0.493
SkiStaza,0.436,0.623,0.942,0.739,0.580,0.490
Skijac,0.427,0.632,0.820,0.863,0.634,0.592
Oprema,0.544,0.607,0.505,0.587,0.776,0.578
Instructor,0.416,0.520,0.440,0.572,0.548,0.880


In [68]:
def match_tables(model, professor: dict, student: dict):
    sim_matrix, prof_tables, stud_tables = build_table_similarity_matrix(
        model, professor, student
    )
    n_p, n_s = len(prof_tables), len(stud_tables)

    # Pad to square matrix so Hungarian algo works with unequal counts
    size = max(n_p, n_s)
    padded = np.zeros((size, size))
    padded[:n_p, :n_s] = sim_matrix

    # Maximise similarity (algorithm minimises, so negate)
    row_ind, col_ind = linear_sum_assignment(-padded)

    matches = []
    matched_prof, matched_stud = set(), set()

    for r, c in zip(row_ind, col_ind):
        if r >= n_p or c >= n_s:
            continue  # padding slot
        score = float(sim_matrix[r, c])
        pt, st = prof_tables[r], stud_tables[c]
        if score >= TABLE_MATCH_THRESHOLD:
            matches.append({'professor_table': pt, 'student_table': st,
                            'similarity': round(score, 3), 'status': '✓ matched'})
            matched_prof.add(pt)
            matched_stud.add(st)
        else:
            matches.append({'professor_table': pt, 'student_table': st,
                            'similarity': round(score, 3), 'status': '⚠ below threshold'})

    # Tables only in professor (missing from student)
    for pt in prof_tables:
        if pt not in matched_prof:
            matches.append({'professor_table': pt, 'student_table': '—',
                            'similarity': 0.0, 'status': '❌ missing in student'})

    # Tables only in student (extra, not in professor)
    for st in stud_tables:
        if st not in matched_stud:
            matches.append({'professor_table': '—', 'student_table': st,
                            'similarity': 0.0, 'status': '➕ extra in student'})

    return matches

for schema in schemas['latin']:
    table_matches = match_tables(model, PROFESSOR_SCHEMA, schema)

    df_tables = pd.DataFrame(table_matches)
    display(df_tables.style.map(
        lambda v: 'color: green; font-weight: bold'  if '✓' in str(v)
             else 'color: red'                        if '❌' in str(v)
             else 'color: orange'                     if '⚠' in str(v) or '➕' in str(v)
             else '',
        subset=['status']
    ).format({'similarity': '{:.3f}'}))
table_matches = match_tables(model, PROFESSOR_SCHEMA, STUDENT_SCHEMA)

,professor_table,student_table,similarity,status
0,SkiKarta,SkiTicket,0.943,✓ matched
1,SkiLift,SkiingGear,0.717,✓ matched
2,SkiStaza,SkiStage,0.959,✓ matched
3,Skijac,Skier,0.933,✓ matched
4,Oprema,Zicara,0.594,✓ matched
5,Instructor,Instruktor,0.920,✓ matched


,professor_table,student_table,similarity,status
0,SkiKarta,Vleznica,0.656,✓ matched
1,SkiLift,Gondola,0.586,✓ matched
2,SkiStaza,Teren,0.713,✓ matched
3,Skijac,Skier,0.901,✓ matched
4,Oprema,Gear,0.876,✓ matched
5,Instructor,Instructor,0.990,✓ matched


,professor_table,student_table,similarity,status
0,SkiKarta,SkiVleznica,0.874,✓ matched
1,SkiLift,Zhichara,0.583,✓ matched
2,SkiStaza,Slope,0.813,✓ matched
3,Skijac,Skijach,0.775,✓ matched
4,Oprema,Gear,0.790,✓ matched
5,Instructor,Instructor,0.970,✓ matched


,professor_table,student_table,similarity,status
0,SkiKarta,SkiTicket,0.911,✓ matched
1,SkiLift,Zicara,0.557,✓ matched
2,SkiStaza,SkiTeren,0.902,✓ matched
3,Skijac,Skijach,0.853,✓ matched
4,Oprema,SkiingGear,0.736,✓ matched
5,Instructor,Tutor,0.895,✓ matched


,professor_table,student_table,similarity,status
0,SkiKarta,LiftPass,0.631,✓ matched
1,SkiLift,Lift,0.811,✓ matched
2,SkiStaza,Teren,0.711,✓ matched
3,Skijac,Skijach,0.857,✓ matched
4,Oprema,Gear,0.771,✓ matched
5,Instructor,Tutor,0.898,✓ matched


...

,professor_table,student_table,similarity,status
0,SkiKarta,LiftPass,0.564,✓ matched
1,SkiLift,SkiingGear,0.771,✓ matched
2,SkiStaza,SkiStage,0.874,✓ matched
3,Skijac,Skier,0.893,✓ matched
4,Oprema,Gondola,0.546,✓ matched
5,Instructor,Tutor,0.855,✓ matched


,professor_table,student_table,similarity,status
0,SkiKarta,LiftPass,0.648,✓ matched
1,SkiLift,Zicara,0.546,✓ matched
2,SkiStaza,SkiStage,0.861,✓ matched
3,Skijac,Skier,0.933,✓ matched
4,Oprema,Equipment,0.886,✓ matched
5,Instructor,Instructor,0.992,✓ matched


,professor_table,student_table,similarity,status
0,SkiKarta,Karta,0.757,✓ matched
1,SkiLift,Gondola,0.523,✓ matched
2,SkiStaza,Slope,0.800,✓ matched
3,Skijac,Skier,0.942,✓ matched
4,Oprema,Gear,0.784,✓ matched
5,Instructor,Tutor,0.895,✓ matched


,professor_table,student_table,similarity,status
0,SkiKarta,SkiTicket,0.872,✓ matched
1,SkiLift,SkiingGear,0.718,✓ matched
2,SkiStaza,SkiStage,0.911,✓ matched
3,Skijac,Skier,0.893,✓ matched
4,Oprema,Gondola,0.564,✓ matched
5,Instructor,Tutor,0.843,✓ matched


,professor_table,student_table,similarity,status
0,SkiKarta,Vleznica,0.558,✓ matched
1,SkiLift,Zicara,0.579,✓ matched
2,SkiStaza,SkiTeren,0.942,✓ matched
3,Skijac,Skijach,0.863,✓ matched
4,Oprema,Gear,0.776,✓ matched
5,Instructor,Tutor,0.880,✓ matched


In [69]:
def match_attributes_for_pair(model, prof_attrs: list, stud_attrs: list) -> list:
    """
    Match attributes using a combined score of:
      - Name embedding cosine similarity  (W_ATTR_NAME)
      - SQL type compatibility             (W_ATTR_TYPE)

    This lets 'vozrast INTEGER' match 'age INTEGER' even when the embedding
    similarity alone is below threshold.
    """
    if not prof_attrs or not stud_attrs:
        return []

    p_names = [a for a, _ in prof_attrs]
    s_names = [a for a, _ in stud_attrs]
    p_types = [t for _, t in prof_attrs]
    s_types = [t for _, t in stud_attrs]

    pe = embed(model, [readable(n) for n in p_names])
    se = embed(model, [readable(n) for n in s_names])

    n_p, n_s = len(prof_attrs), len(stud_attrs)

    # Build combined similarity matrix
    sim_matrix = np.zeros((n_p, n_s))
    for i, j in product(range(n_p), range(n_s)):
        name_sim = cosine(pe[i], se[j])
        type_sim = type_compatible(p_types[i], s_types[j])
        sim_matrix[i, j] = W_ATTR_NAME * name_sim + W_ATTR_TYPE * type_sim

    size = max(n_p, n_s)
    padded = np.zeros((size, size))
    padded[:n_p, :n_s] = sim_matrix
    row_ind, col_ind = linear_sum_assignment(-padded)

    results = []
    matched_p, matched_s = set(), set()

    for r, c in zip(row_ind, col_ind):
        if r >= n_p or c >= n_s:
            continue
        score = float(sim_matrix[r, c])
        pa, pt_type = prof_attrs[r]
        sa, st_type = stud_attrs[c]
        if score >= ATTR_MATCH_THRESHOLD:
            type_ok = '✓' if normalise_type(pt_type) == normalise_type(st_type) else '⚠ type mismatch'
            results.append({
                'professor_attr': pa, 'prof_type': pt_type,
                'student_attr':   sa, 'stud_type': st_type,
                'similarity': round(score, 3),
                'type_check': type_ok,
                'status': '✓ matched'
            })
            matched_p.add(pa); matched_s.add(sa)

    for pa, pt_type in prof_attrs:
        if pa not in matched_p:
            results.append({'professor_attr': pa, 'prof_type': pt_type,
                            'student_attr': '—', 'stud_type': '—',
                            'similarity': 0.0, 'type_check': '—', 'status': '❌ missing'})
    for sa, st_type in stud_attrs:
        if sa not in matched_s:
            results.append({'professor_attr': '—', 'prof_type': '—',
                            'student_attr': sa, 'stud_type': st_type,
                            'similarity': 0.0, 'type_check': '—', 'status': '➕ extra'})
    return results


attr_matches_map = {}

for m in table_matches:
    if m['status'] != '✓ matched':
        continue
    pt, st = m['professor_table'], m['student_table']
    attr_result = match_attributes_for_pair(
        model,
        PROFESSOR_SCHEMA.get(pt, []),
        STUDENT_SCHEMA.get(st, [])
    )
    attr_matches_map[pt] = attr_result

    print(f'\n── {st}  ↔  {pt} ──────────────────────')
    df_attrs = pd.DataFrame(attr_result)
    display(df_attrs.style.map(
        lambda v: 'color: green; font-weight: bold' if v == '✓ matched'
             else 'color: red'                      if '❌' in str(v)
             else 'color: orange'                   if '➕' in str(v) or 'mismatch' in str(v)
             else '',
        subset=['status', 'type_check']
    ).format({'similarity': '{:.3f}'}))



── SkiPass  ↔  SkiKarta ──────────────────────


,professor_attr,prof_type,student_attr,stud_type,similarity,type_check,status
0,cena,NUMERIC,price,NUMERIC,0.640,✓,✓ matched
1,vazi_od,DATE,valid_from,DATE,0.624,✓,✓ matched
2,vazi_do,DATE,valid_to,DATE,0.686,✓,✓ matched
3,skijac_id,INTEGER,skier_id,INTEGER,0.970,✓,✓ matched



── SkiLift  ↔  SkiLift ──────────────────────


,professor_attr,prof_type,student_attr,stud_type,similarity,type_check,status
0,ime,TEXT,name,TEXT,0.838,✓,✓ matched
1,kapacitet,INTEGER,capacity,INTEGER,0.990,✓,✓ matched
2,dolzina,NUMERIC,length,NUMERIC,0.553,✓,✓ matched
3,status,TEXT,status,TEXT,1.000,✓,✓ matched



── SkiSlope  ↔  SkiStaza ──────────────────────


...

,professor_attr,prof_type,student_attr,stud_type,similarity,type_check,status
0,ime,TEXT,name,TEXT,0.838,✓,✓ matched
1,prezime,TEXT,last_name,TEXT,0.550,✓,✓ matched
2,vozrast,INTEGER,age,INTEGER,0.630,✓,✓ matched
3,nivo_na_veshtina,TEXT,skill_level,TEXT,0.685,✓,✓ matched



── SkiEquipment  ↔  Oprema ──────────────────────


,professor_attr,prof_type,student_attr,stud_type,similarity,type_check,status
0,tip,TEXT,type,TEXT,0.658,✓,✓ matched
1,golemina,TEXT,size,TEXT,0.571,✓,✓ matched
2,brend,TEXT,brand,TEXT,0.916,✓,✓ matched
3,status,TEXT,status,TEXT,1.000,✓,✓ matched



── Instructor  ↔  Instructor ──────────────────────


,professor_attr,prof_type,student_attr,stud_type,similarity,type_check,status
0,ime,TEXT,surname,TEXT,0.858,✓,✓ matched
1,prezime,TEXT,name,TEXT,0.662,✓,✓ matched
2,nivo_na_licenca,TEXT,license_level,TEXT,0.954,✓,✓ matched
3,godini_iskustvo,INTEGER,experience_years,INTEGER,0.939,✓,✓ matched


In [70]:
def compute_scores(professor, table_matches, attr_matches_map):
    n_prof_tables = len(professor)
    matched_tables = sum(1 for m in table_matches if m['status'] == '✓ matched')
    table_score = matched_tables / n_prof_tables if n_prof_tables else 0

    total_prof_attrs = sum(len(v) for v in professor.values())
    matched_attrs      = sum(sum(1 for a in al if a['status'] == '✓ matched') for al in attr_matches_map.values())
    type_mismatches    = sum(sum(1 for a in al if 'mismatch' in str(a.get('type_check',''))) for al in attr_matches_map.values())
    attr_score = matched_attrs / total_prof_attrs if total_prof_attrs else 0
    overall = 0.40 * table_score + 0.60 * attr_score

    return {
        'Tables matched':    f'{matched_tables} / {n_prof_tables}',
        'Table score':       f'{table_score*100:.1f}%',
        'Attrs matched':     f'{matched_attrs} / {total_prof_attrs}',
        'Type mismatches':   str(type_mismatches),
        'Attribute score':   f'{attr_score*100:.1f}%',
        'Overall score':     f'{overall*100:.1f}%',
    }


scores = compute_scores(PROFESSOR_SCHEMA, table_matches, attr_matches_map)
df_scores = pd.DataFrame(list(scores.items()), columns=['Metric', 'Value'])
display(df_scores.style.set_properties(**{'font-size': '14pt'}).hide(axis='index'))
print(f"\n🎯 Overall score: {scores['Overall score']}")


Metric,Value
Tables matched,6 / 6
Table score,100.0%
Attrs matched,24 / 24
Type mismatches,0
Attribute score,100.0%
Overall score,100.0%



🎯 Overall score: 100.0%


In [71]:
result = {
    'table_matches': table_matches,
    'attribute_matches': {
        pt: attr_list for pt, attr_list in attr_matches_map.items()
    },
    'scores': scores,
}

with open('match_result.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print('Results saved to match_result.json ✓')

Results saved to match_result.json ✓


In [72]:
import numpy as np

def prepare_label_cache(model, labels):
    labels_readable = [readable(l) for l in labels]
    label_emb = embed(model, labels_readable)
    label_norms = np.linalg.norm(label_emb, axis=1) + 1e-12
    return labels_readable, label_emb, label_norms

def match_word_to_labels(model, word, labels, label_emb, label_norms, top_k=5, threshold=0.40):
    # threshold=0.40 matches the style of your TABLE_MATCH_THRESHOLD :contentReference[oaicite:3]{index=3}
    w = embed(model, [readable(word)])[0]
    w_norm = np.linalg.norm(w) + 1e-12

    sims = (label_emb @ w) / (label_norms * w_norm)  # cosine, vectorized
    order = np.argsort(-sims)

    best_i = int(order[0])
    best = (labels[best_i], float(sims[best_i]))
    accepted = best[1] >= threshold

    top = [(labels[int(i)], float(sims[int(i)])) for i in order[:top_k]]
    return {"best": best, "accepted": accepted, "top": top}

labels = ["company","hackathon","held_hackathon","engineer",
          "company_engineer","sponsor_company","company_engineer_held_hackathon"]
labels_readable, label_emb, label_norms = prepare_label_cache(model, labels)

print(match_word_to_labels(model, "хакатон", labels, label_emb, label_norms, top_k=3, threshold=0.40))
print(match_word_to_labels(model, "SkiPass", labels, label_emb, label_norms, top_k=3, threshold=0.40))
# ile e peder

{'best': ('company', 0.41137829422950745), 'accepted': True, 'top': [('company', 0.41137829422950745), ('hackathon', 0.36209288239479065), ('engineer', 0.33575817942619324)]}
{'best': ('sponsor_company', 0.12316960096359253), 'accepted': False, 'top': [('sponsor_company', 0.12316960096359253), ('company', 0.1148960068821907), ('held_hackathon', 0.06687742471694946)]}
